In [28]:
!pip install -qU mcp langchain langchain-mcp-adapters langchain-groq uvicorn

In [29]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY_3")

print("✅ Groq API key loaded")

✅ Groq API key loaded


Upload database.db

In [30]:
from google.colab import files
import os

print("📂 Upload your database.db file")

uploaded = files.upload()

print("\nChecking database...")

if os.path.exists("/content/database.db"):
    print("✅ database.db uploaded successfully")
else:
    print("❌ database.db not found")

📂 Upload your database.db file


Saving database.db to database (2).db

Checking database...
✅ database.db uploaded successfully


Check database

In [31]:
import sqlite3

DATABASE = "/content/database.db"

conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""")

tables = cursor.fetchall()

print("======================================")
print("📊 DATABASE TABLES")
print("======================================")

for table in tables:
    print(f" - {table[0]}")

conn.close()

📊 DATABASE TABLES
 - inventory
 - sqlite_sequence
 - users
 - vendors


Create MCP SQLite Server

In [32]:
%%writefile sqlite_mcp_server.py

import sqlite3

from mcp.server.fastmcp import FastMCP


# =========================================================
# DATABASE
# =========================================================

DATABASE = "/content/database.db"


# =========================================================
# MCP SERVER
# =========================================================

mcp = FastMCP(
    "SQLite Database Server",
    stateless_http=True,
    json_response=True
)


# =========================================================
# TOOL 1 — LIST TABLES
# =========================================================

@mcp.tool()
def list_tables() -> str:
    """List all tables in the SQLite database."""

    try:

        with sqlite3.connect(DATABASE) as conn:

            cursor = conn.cursor()

            cursor.execute("""
                SELECT name
                FROM sqlite_master
                WHERE type='table'
                ORDER BY name
            """)

            tables = cursor.fetchall()

        if not tables:
            return "No tables found."

        return "\n".join(
            table[0]
            for table in tables
        )

    except Exception as e:

        return f"Database error: {e}"


# =========================================================
# TOOL 2 — DESCRIBE TABLE
# =========================================================

@mcp.tool()
def describe_table(table_name: str) -> str:
    """Show the columns and data types of a table."""

    try:

        with sqlite3.connect(DATABASE) as conn:

            cursor = conn.cursor()

            cursor.execute(
                """
                SELECT name
                FROM sqlite_master
                WHERE type='table'
                AND name=?
                """,
                (table_name,)
            )

            table = cursor.fetchone()

            if not table:

                return (
                    f"Table '{table_name}' "
                    f"does not exist."
                )

            cursor.execute(
                f'PRAGMA table_info("{table_name}")'
            )

            columns = cursor.fetchall()

        result = []

        result.append(
            f"Table: {table_name}"
        )

        result.append("")

        result.append(
            "Column | Data Type"
        )

        result.append(
            "------------------"
        )

        for column in columns:

            result.append(
                f"{column[1]} | {column[2]}"
            )

        return "\n".join(result)

    except Exception as e:

        return f"Database error: {e}"


# =========================================================
# TOOL 3 — READ QUERY
# =========================================================

@mcp.tool()
def read_query(query: str) -> str:
    """
    Execute a read-only SQL query.
    Returns the complete result.
    """

    try:

        query = query.strip()

        if not query:

            return "Query is empty."

        clean_query = query.rstrip(";").strip()

        query_lower = clean_query.lower()

        # Only read operations
        allowed = (
            "select",
            "pragma",
            "with"
        )

        if not query_lower.startswith(allowed):

            return (
                "Only SELECT, PRAGMA, "
                "and WITH queries are allowed."
            )

        # Block write operations
        blocked = [
            "insert ",
            "update ",
            "delete ",
            "drop ",
            "alter ",
            "create ",
            "replace ",
            "attach ",
            "detach "
        ]

        for keyword in blocked:

            if keyword in query_lower:

                return (
                    f"Blocked SQL operation: "
                    f"{keyword.strip()}"
                )

        # Execute query
        with sqlite3.connect(DATABASE) as conn:

            cursor = conn.cursor()

            cursor.execute(clean_query)

            rows = cursor.fetchall()

            if cursor.description is None:

                return "Query returned no columns."

            columns = [
                description[0]
                for description in cursor.description
            ]

        if not rows:

            return "Query returned 0 rows."

        # Build table
        result = []

        # Header
        result.append(
            " | ".join(columns)
        )

        # Separator
        result.append(
            " | ".join(
                "-" * max(3, len(column))
                for column in columns
            )
        )

        # Data
        for row in rows:

            result.append(
                " | ".join(
                    "NULL"
                    if value is None
                    else str(value)
                    for value in row
                )
            )

        result.append("")

        result.append(
            f"Total rows: {len(rows)}"
        )

        return "\n".join(result)

    except sqlite3.Error as e:

        return (
            f"SQLite Error: "
            f"{type(e).__name__}: {e}"
        )

    except Exception as e:

        return (
            f"Unexpected Error: "
            f"{type(e).__name__}: {e}"
        )


# =========================================================
# TOOL 4 — DATABASE INFO
# =========================================================

@mcp.tool()
def database_info() -> str:
    """Show database information."""

    try:

        with sqlite3.connect(DATABASE) as conn:

            cursor = conn.cursor()

            cursor.execute("""
                SELECT name
                FROM sqlite_master
                WHERE type='table'
                ORDER BY name
            """)

            tables = cursor.fetchall()

        result = []

        result.append(
            f"Database: {DATABASE}"
        )

        result.append(
            f"Number of tables: {len(tables)}"
        )

        result.append("")

        result.append("Tables:")

        for table in tables:

            result.append(
                f"- {table[0]}"
            )

        return "\n".join(result)

    except Exception as e:

        return f"Database error: {e}"


# =========================================================
# START SERVER
# =========================================================

if __name__ == "__main__":

    mcp.run(
        transport="streamable-http"
    )

Overwriting sqlite_mcp_server.py


Start MCP Server

In [33]:
import subprocess
import sys
import time

server_process = subprocess.Popen(
    [
        sys.executable,
        "sqlite_mcp_server.py"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

time.sleep(3)

if server_process.poll() is None:

    print("======================================")
    print("✅ MCP SQLite Server Started")
    print("======================================")
    print()
    print("🌐 MCP URL:")
    print("http://127.0.0.1:8000/mcp")

else:

    print("❌ MCP server failed to start")

    error = server_process.stderr.read().decode(
        errors="replace"
    )

    print(error)

✅ MCP SQLite Server Started

🌐 MCP URL:
http://127.0.0.1:8000/mcp


Test MCP

In [34]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def test_mcp():

    async with streamable_http_client(
        "http://127.0.0.1:8000/mcp"
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            print(
                "✅ MCP Server connected"
            )

            tools = await session.list_tools()

            print(
                "\n🔧 MCP Tools loaded:"
            )

            for tool in tools.tools:

                print(
                    f" - {tool.name}"
                )


await test_mcp()

✅ MCP Server connected

🔧 MCP Tools loaded:
 - list_tables
 - describe_table
 - read_query
 - database_info


Test Full Inventory Table

In [35]:
async def test_inventory():

    async with streamable_http_client(
        "http://127.0.0.1:8000/mcp"
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "read_query",
                {
                    "query":
                    "SELECT * FROM inventory"
                }
            )

            print(
                "======================================"
            )

            print(
                "📦 COMPLETE INVENTORY TABLE"
            )

            print(
                "======================================"
            )

            for content in result.content:

                if hasattr(content, "text"):

                    print(
                        content.text
                    )


await test_inventory()

📦 COMPLETE INVENTORY TABLE
item_id | item_name | quantity | price | category
------- | --------- | -------- | ----- | --------
1 | Laptop | 25 | 899.99 | Electronics
2 | Desk Chair | 15 | 129.5 | Furniture
3 | Notebook | 100 | 4.99 | Office Supplies
4 | Coffee Maker | 8 | 59.99 | Kitchen
5 | Wireless Mouse | 30 | 24.99 | Electronics
6 | Desk Lamp | 12 | 34.5 | Lighting
7 | Printer Paper | 200 | 9.99 | Office Supplies
8 | Bluetooth Speaker | 18 | 79.99 | Electronics
9 | Standing Desk | 5 | 299.99 | Furniture
10 | Whiteboard | 10 | 45 | Office Supplies

Total rows: 10


Initialize Groq

In [36]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("✅ Groq model initialized")

✅ Groq model initialized


Load MCP Tools

In [37]:
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent


async def create_database_agent():

    mcp_context = streamable_http_client(
        "http://127.0.0.1:8000/mcp"
    )

    return mcp_context

Complete Agent

In [38]:
async def run_database_assistant():

    async with streamable_http_client(
        "http://127.0.0.1:8000/mcp"
    ) as (
        read_stream,
        write_stream,
        _
    ):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            # =============================================
            # MCP INITIALIZATION
            # =============================================

            await session.initialize()

            print(
                "\n✅ MCP Server connected"
            )

            # =============================================
            # LOAD MCP TOOLS
            # =============================================

            tools = await load_mcp_tools(
                session
            )

            print(
                "\n🔧 MCP Tools loaded:"
            )

            for tool in tools:

                print(
                    f" - {tool.name}"
                )

            # =============================================
            # CREATE AGENT
            # =============================================

            agent = create_agent(

                model,

                tools,

                system_prompt="""

You are a SQLite Database Assistant.

You have access to a SQLite database
through MCP tools.

IMPORTANT:

When the user asks for actual data,
you MUST use the read_query tool.

When the user asks:

- show all data
- show all items
- show all records
- read all data
- display all rows
- show complete table
- show full table

you MUST retrieve the actual rows.

Do NOT summarize the data.

Do NOT describe only the columns.

Do NOT use describe_table for an
all-data request.

For example:

User:
Show all data from inventory.

Use:

SELECT * FROM inventory

through read_query.

If the user asks for table structure,
use describe_table.

If the user asks for available tables,
use list_tables.

If the user asks for general database
information, use database_info.

Never invent database results.

"""
            )

            # =============================================
            # START CHAT
            # =============================================

            print(
                "\n======================================"
            )

            print(
                "🤖 SQLite Database Assistant"
            )

            print(
                "🚀 Groq + LangChain + MCP"
            )

            print(
                "======================================"
            )

            print(
                "Type 'exit' to quit"
            )

            # =============================================
            # CHAT LOOP
            # =============================================

            while True:

                query = input(
                    "\nEnter your query: "
                ).strip()

                # -----------------------------------------
                # EXIT
                # -----------------------------------------

                if query.lower() == "exit":

                    print(
                        "\n👋 Goodbye!"
                    )

                    break

                # -----------------------------------------
                # EMPTY
                # -----------------------------------------

                if not query:

                    continue

                # -----------------------------------------
                # PROCESS
                # -----------------------------------------

                print(
                    "\n⏳ Processing..."
                )

                try:

                    response = await agent.ainvoke(
                        {
                            "messages": [
                                {
                                    "role": "user",
                                    "content": query
                                }
                            ]
                        }
                    )

                    print(
                        "\n🤖 Answer:"
                    )

                    print(
                        response[
                            "messages"
                        ][-1].content
                    )

                except Exception as e:

                    print(
                        "\n❌ Query Error"
                    )

                    print(
                        f"Error Type: "
                        f"{type(e).__name__}"
                    )

                    print(
                        f"Error: {e}"
                    )

In [ ]:
await run_database_assistant()


✅ MCP Server connected

🔧 MCP Tools loaded:
 - list_tables
 - describe_table
 - read_query
 - database_info

🤖 SQLite Database Assistant
🚀 Groq + LangChain + MCP
Type 'exit' to quit

Enter your query: show all the tables

⏳ Processing...

🤖 Answer:
These are the tables in your SQLite database: inventory, sqlite_sequence, users, vendors.

Enter your query: show the structure of  vendors

⏳ Processing...

🤖 Answer:
The structure of the 'vendors' table is as follows:

- VendorID: INTEGER
- VendorName: TEXT
- ContactEmail: TEXT
- PhoneNumber: TEXT
- Address: TEXT

Enter your query: read all the data from the inventory table and display the complete table with all rows and all columns. Do not summarize.

⏳ Processing...

🤖 Answer:
The inventory table has 10 rows and 5 columns. The columns are item_id, item_name, quantity, price, and category. The data types of these columns are integer, text, integer, real, and text respectively. 

Here is the data:
item_id | item_name | quantity | price |


      ✅ MCP Server connected

      🔧 MCP Tools loaded:
      - list_tables
      - describe_table
      - read_query
      - database_info

      ======================================
      🤖 SQLite Database Assistant
      🚀 Groq + LangChain + MCP
      ======================================
      Type 'exit' to quit

      Enter your query: how many items are in inventory?

      ⏳ Processing...

      🤖 Answer:
      There are 10 items in the inventory.

      Enter your query: show inventory items where quantity is less than 10

      ⏳ Processing...

      🤖 Answer:
      The inventory items where the quantity is less than 10 are:

      * Coffee Maker (item_id: 4, quantity: 8, price: 59.99, category: Kitchen)
      * Standing Desk (item_id: 9, quantity: 5, price: 299.99, category: Furniture)

      Enter your query: show all tables

      ⏳ Processing...

      ❌ Query Error
      Error Type: RateLimitError
      Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kwpv7mvdepcrwjrs1r3cqdgm` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99687, Requested 599. Please try again in 4m7.104s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

      Enter your query: exit

      👋 Goodbye!


      ✅ MCP Server connected

      🔧 MCP Tools loaded:
      - list_tables
      - describe_table
      - read_query
      - database_info

      ======================================
      🤖 SQLite Database Assistant
      🚀 Groq + LangChain + MCP
      ======================================
      Type 'exit' to quit

      Enter your query: show all the tables

      ⏳ Processing...

      🤖 Answer:
      These are the tables in your SQLite database: inventory, sqlite_sequence, users, vendors.

      Enter your query: show the structure of  vendors

      ⏳ Processing...

      🤖 Answer:
      The structure of the 'vendors' table is as follows:

      - VendorID: INTEGER
      - VendorName: TEXT
      - ContactEmail: TEXT
      - PhoneNumber: TEXT
      - Address: TEXT

      Enter your query: read all the data from the inventory table and display the complete table with all rows and all columns. Do not summarize.

      ⏳ Processing...

      🤖 Answer:
      The inventory table has 10 rows and 5 columns. The columns are item_id, item_name, quantity, price, and category. The data types of these columns are integer, text, integer, real, and text respectively.

      Here is the data:
      item_id | item_name | quantity | price | category
      ------- | --------- | -------- | ----- | --------
      1 | Laptop | 25 | 899.99 | Electronics
      2 | Desk Chair | 15 | 129.5 | Furniture
      3 | Notebook | 100 | 4.99 | Office Supplies
      4 | Coffee Maker | 8 | 59.99 | Kitchen
      5 | Wireless Mouse | 30 | 24.99 | Electronics
      6 | Desk Lamp | 12 | 34.5 | Lighting
      7 | Printer Paper | 200 | 9.99 | Office Supplies
      8 | Bluetooth Speaker | 18 | 79.99 | Electronics
      9 | Standing Desk | 5 | 299.99 | Furniture
      10 | Whiteboard | 10 | 45 | Office Supplies

      Total rows: 10

      Enter your query: show inventory items where quantity is less than 10

      ⏳ Processing...